# GRU（門控循環單元，2014）

- **核心原理**：精簡為更新、重置兩個閘門
- **推薦資料集**：IMDB 影評情感分類（與第 3 課 LSTM 比較；同一份資料、同樣的前處理與切分）
- **論文**：Cho et al. (2014), *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*；Chung et al. (2014), *Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling*

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 3 課（LSTM）的差別：

| | 第 3 課 LSTM | 本章 GRU |
| --- | --- | --- |
| 狀態 | 兩個：$c_t$（記憶）、$h_t$（輸出） | **一個** $h_t$（記憶就是輸出） |
| 閘門 | 輸入 $i$、遺忘 $f$、輸出 $o$ + 候選 $g$（4 段） | **更新** $z$、**重置** $r$ + 候選 $n$（3 段） |
| 更新方式 | $c_t = f \odot c_{t-1} + i \odot g$（$f$、$i$ 各自獨立） | $h_t = z \odot h_{t-1} + (1 - z)\odot n$（「留多少」和「寫多少」**綁在一起**，加起來 = 1） |
| 參數量（循環層） | $4H(E + H) + \ldots$ | $3H(E + H) + \ldots$（約少 25%） |
| 梯度的捷徑 | 沿著 $c$：每一步乘 $f_t$ | 沿著 $h$：每一步乘 $z_t$（再加上經過閘門的部分） |

一句話：**GRU 把 LSTM 的「遺忘」和「輸入」合成一個更新閘門 $z$（新舊內容按比例混合），拿掉輸出閘門與獨立的 cell state —— 更少的參數，同樣有「梯度捷徑」。**

這一章的重點：**GRU 的公式（以及 PyTorch 版本跟原論文的一個差異）、它的反向傳播、更新閘門 = 插值、GRU vs LSTM 在相同條件／相同參數量下誰比較好，以及怎麼判斷兩個模型的差距是不是真的（seed 雜訊、配對 bootstrap）。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5、6 節開頭另有各自的符號表或名詞表。

**資料**：跟第 3 課完全一樣 —— `load_imdb()`、去掉 `<br />`、轉小寫、切字；字表 = train（20,000 篇）最常見的 20,000 個字（0 = padding、1 = `<unk>`）；每篇保留**最後** `MAX_LEN = 200` 個字；validation 5,000 篇（同一個 `random_state`，所以跟第 3 課是同一批）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | shape／本章的值 |
| --- | --- | --- |
| $n$、$T$ | batch 大小、batch 裡最長的長度 | 64、≤ 200 |
| $V$、$E$、$H$ | 字表大小、字向量維度、隱藏維度 | 20,002、64、64 |
| $r_t$ | 重置閘門（reset，sigmoid）：計算候選內容時，舊的 $h$ 要用多少 | $(n, H)$ |
| $z_t$ | 更新閘門（update，sigmoid）：新的 $h$ 有多少比例**沿用**舊的 $h$ | $(n, H)$ |
| $n_t$ | 候選內容（tanh） | $(n, H)$ |
| $h_t$ | 隱藏狀態；$h_0 = 0$ | $(n, H)$ |
| $W_x$、$W_h$ | 三段並排 $[r \mid z \mid n]$ | $(E, 3H)$、$(H, 3H)$ |
| $b$、$b_{hn}$ | 輸入側的偏差 $(3H,)$；候選內容「$h$ 那一側」的偏差 $(H,)$（見 3A） | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 課 | MLP＋反向傳播 | `Linear` 的梯度 $X^\top\delta$；softmax＋交叉熵 $(P - Y)/n$；梯度檢查（相對誤差 < 1e-6）。 |
| 第 2 課 | RNN（`03-early-nn/02-rnn`） | `Embedding`（`np.add.at`）、mask（padding 保持舊的 $h$）、BPTT 每一步乘 $(1 - h^2)W_h^\top$ → 消失或爆炸；梯度裁剪。 |
| 第 3 課 | LSTM（`03-early-nn/03-lstm`），同一份 IMDB | 四個閘門、$c_t = f \odot c_{t-1} + i \odot g$；沿著 $c$ 往回每一步只乘 $f_t$（constant error carousel）。結果：PyTorch RNN val 0.74、LSTM 0.86；TF-IDF + LogReg 0.90（test 0.911）沒被打敗；留結尾比留開頭好（0.863 vs 0.833）；3 個 seed 的 val accuracy 差到約 0.02；手寫 NumPy 用 Adam 在 8,000 篇子集上訓練。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 凸組合（插值）**：$z a + (1 - z) b$，$0 \le z \le 1$ → 結果在 $a$ 和 $b$ 之間。$z = 1$ 完全是 $a$、$z = 0$ 完全是 $b$。例：$z = 0.7$、$a = 0$、$b = 1$ → $0.3$。GRU 的 $h_t = z_t \odot h_{t-1} + (1 - z_t) \odot n_t$ 就是逐維度的插值。

**2. 插值的導數**：$\frac{\partial}{\partial z}[z a + (1 - z) b] = a - b$；對 $a$ 是 $z$、對 $b$ 是 $1 - z$。

**3. sigmoid、tanh 的導數**（第 3 課）：$\sigma' = \sigma(1 - \sigma)$、$\tanh' = 1 - \tanh^2$。

**4. 標準誤與信賴區間**：比率 $p$ 在 $n$ 個樣本上的標準誤 $\approx \sqrt{p(1-p)/n}$。例：$p = 0.87$、$n = 25{,}000$ → 0.0021。95% 信賴區間 ≈ 估計值 ± 2 個標準誤。

**5. 配對比較**：兩個模型在**同一批**測試樣本上，與其各自算 accuracy 再比，不如看「每一篇誰對誰錯」—— 兩個模型大部分的對錯是一樣的，真正有資訊的只有「意見不同」的那些篇。**bootstrap**：從測試集**有放回地**抽 $n$ 篇，重新算差距，重複很多次 → 差距的分布 → 信賴區間。

#### 程式

**6. 三段並排**：`z = x @ Wx + h @ Wh + b`，切成 `[:, :H]`、`[:, H:2*H]`、`[:, 2*H:]`（第 3 課是四段）。
**7. `np.random.default_rng(seed).integers(0, n, size=(B, n))`**：一次產生 $B$ 組 bootstrap 的索引。

---
## 1. 資料與探索性資料分析（EDA）

資料跟第 3 課一樣，主要的 EDA 結論重述如下（不必重做）：

- train／test 各 25,000 篇，正負面各半 → accuracy 適合當主要指標（另看 log loss、ROC AUC）。
- 字數中位數約 190、99% 分位數約 1,000；只留 200 個字時約 47% 的影評被截斷。
- 前 20,000 個字涵蓋約 97% 的字數；最能區分的字是 waste、worst、awful vs. delightful、superb、gem 這類。
- 同樣 $k$ 個字，**結尾**比開頭更能判斷正負面（詞袋 + LogReg：$k = 200$ 時 0.852 vs 0.832）。

這一章的 EDA 只看一件新的事：**情緒字離結尾多遠？** 如果大部分的線索都在最後幾十個字，那「能記住很遠的東西」（LSTM、GRU 的強項）在這個任務上就沒那麼重要。

In [ ]:
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from learning_map.data import load_imdb

ds = load_imdb()
train_text = [ex["text"] for ex in ds["train"]]
y_full = np.array(ds["train"]["label"])
test_text = [ex["text"] for ex in ds["test"]]
y_test = np.array(ds["test"]["label"])


def tokenize(s: str) -> list[str]:
    """同第 3 課：去掉 <br />、轉小寫，切成字與標點。"""
    return re.findall(r"[a-z0-9']+|[!?.,]", re.sub(r"<br\s*/?>", " ", s.lower()))


train_tok = [tokenize(s) for s in train_text]
test_tok = [tokenize(s) for s in test_text]
VOCAB_SIZE, MAX_LEN = 20_000, 200
tr_idx, va_idx = train_test_split(np.arange(len(train_tok)), test_size=5_000, stratify=y_full, random_state=0)
y_tr, y_va = y_full[tr_idx], y_full[va_idx]
tok_tr = [train_tok[i] for i in tr_idx]
tok_va = [train_tok[i] for i in va_idx]
WORD_TO_ID = {w: k + 2 for k, (w, _) in enumerate(Counter(w for t in tok_tr for w in t).most_common(VOCAB_SIZE))}
ID_TO_WORD = {k: w for w, k in WORD_TO_ID.items()} | {1: "<unk>", 0: "<pad>"}
V = VOCAB_SIZE + 2


def encode(token_lists: list[list[str]], max_len: int = MAX_LEN, side: str = "tail") -> tuple[np.ndarray, np.ndarray]:
    """同第 3 課：→ (字的編號 (n, max_len)，真正長度 (n,))。預設留結尾。"""
    Xi = np.zeros((len(token_lists), max_len), dtype=np.int64)
    lengths = np.zeros(len(token_lists), dtype=np.int64)
    for r, toks in enumerate(token_lists):
        ids = [WORD_TO_ID.get(w, 1) for w in toks]
        ids = ids[:max_len] if side == "head" else ids[-max_len:]
        Xi[r, : len(ids)] = ids
        lengths[r] = max(len(ids), 1)
    return Xi, lengths


Xi_tr, len_tr = encode(tok_tr)
Xi_va, len_va = encode(tok_va)
Xi_te, len_te = encode(test_tok)
print("train / val / test：", len(tok_tr), len(tok_va), len(test_tok), "；Xi_tr", Xi_tr.shape)

In [ ]:
# 情緒字離結尾多遠？
# 觀察：
#   - 「強烈的情緒字」最後一次出現的位置，離結尾的距離的中位數是多少？有多少比例在最後 50 個字內？最後 200 個字內？
#   - 「最後一個情緒字」的正負，跟整篇的標籤一致的比例是多少？
POS_WORDS = {"great", "excellent", "wonderful", "superb", "brilliant", "loved", "perfect", "best", "amazing", "masterpiece", "gem", "delightful"}
NEG_WORDS = {"bad", "worst", "awful", "terrible", "waste", "boring", "horrible", "poor", "stupid", "worse", "dull", "pointless"}
dist, agree = [], []
for toks, lab in zip(tok_tr, y_tr):
    pos = [k for k, w in enumerate(toks) if w in POS_WORDS or w in NEG_WORDS]
    if pos:
        last = pos[-1]
        dist.append(len(toks) - 1 - last)
        agree.append((toks[last] in POS_WORDS) == bool(lab))
dist = np.array(dist)
print(f"有情緒字的影評：{len(dist) / len(tok_tr):.1%}；最後一個情緒字離結尾的距離：中位數 {np.median(dist):.0f}、"
      f"在最後 50 字內 {np.mean(dist < 50):.1%}、最後 200 字內 {np.mean(dist < 200):.1%}")
print(f"「最後一個情緒字」的正負 = 標籤 的比例：{np.mean(agree):.3f}")
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(np.minimum(dist, 400), bins=80)
ax.axvline(200, color="gray", ls=":")
ax.set_xlabel("distance of the last sentiment word from the end (clipped at 400)")
plt.show()

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 情緒字的位置：最後一個情緒字離結尾的中位數距離是多少？在最後 200 個字內的比例？
#     如果模型只要「記住最近 50 個字」就能看到大部分的線索，那 LSTM／GRU「長期記憶」的優勢在這個任務上會大還是小？
# 答：
#
# (2) 「最後一個情緒字」單獨拿來猜，準確率是多少？跟 TF-IDF（0.90）差多少？剩下的差距需要什麼資訊？（否定、轉折、整體的比重…）
# 答：

---
## 2. Baseline

| Baseline | 做法 |
| --- | --- |
| Dummy | 永遠 0.5 |
| TF-IDF 1～2-gram + LogReg | 第 3 課的強 baseline（整篇） |
| **LSTM**（PyTorch，第 3 課的設定） | $E = H = 64$、最後 200 個字、Adam 2e-3、5 epochs、`restore_best` —— **本章主要的比較對象**，第 4 節訓練 |

**評估指標**：accuracy、log loss、ROC AUC（`val_table` 一路往下加）。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline


def val_scores(prob: np.ndarray, y: np.ndarray = y_va) -> dict[str, float]:
    """prob (n, 2)。回傳 {"accuracy", "log_loss", "roc_auc"}。"""
    return {
        "accuracy": round(accuracy_score(y, prob.argmax(axis=1)), 4),
        "log_loss": round(log_loss(y, np.clip(prob, 1e-12, 1), labels=[0, 1]), 4),
        "roc_auc": round(roc_auc_score(y, prob[:, 1]), 4),
    }


docs_tr = [" ".join(t) for t in tok_tr]
docs_va = [" ".join(t) for t in tok_va]
val_table = pd.DataFrame(columns=["accuracy", "log_loss", "roc_auc"])
val_table.loc["Dummy prior"] = val_scores(np.full((len(y_va), 2), 0.5))
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, token_pattern=r"\S+"), LogisticRegression(C=4, max_iter=3000)).fit(docs_tr, y_tr)
val_table.loc["TF-IDF 1-2-gram + LogReg"] = val_scores(tfidf.predict_proba(docs_va))
display(val_table)

In [ ]:
# TODO 2（問答）先寫下預測（第 4～6 節驗證，寫錯沒關係）
#
# (1) 同樣的 E、H、epoch，GRU 的 validation accuracy 會比 LSTM 高、低、還是差不多？訓練速度呢？
# (2) 第 3 課 3 個 seed 的 LSTM val accuracy 差到約 0.02。如果 GRU 和 LSTM 只差 0.005，你能說誰比較好嗎？要怎麼判斷？
# 答：

---
## 3. 手寫模型（NumPy）

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | `GRU` 的 forward：重置閘門、更新閘門、候選內容、插值 | 3、4 |
| 3B | GRU 的反向傳播 | 5（推導）、6 |
| 3C | 梯度檢查；跟 RNN、LSTM 一起在**子集**上訓練 | 7 |
| 3D | 實驗：梯度能傳多遠、更新閘門在做什麼、逐字的預測 | 8 |

### 從第 1～3 課搬過來的層（已完成）

- 第 1 課：`Linear`、`SoftmaxCrossEntropy`、`numerical_grad`、`rel_error` 等；
- 第 2 課：`Embedding`、`RNN`、`clip_grad_norm`；
- 第 3 課：`sigmoid`、`LSTM`（含 `dh_norms`、`dc_norms`）。

原封不動。

In [ ]:
from typing import Literal, Self
from numpy.typing import ArrayLike
from scipy.special import logsumexp

InitName = Literal["he", "xavier"]


def init_std(init: InitName | float, fan_in: int, fan_out: int) -> float:
    """權重初始化的標準差。He：sqrt(2 / fan_in)（給 ReLU）；Xavier：sqrt(2 / (fan_in + fan_out))（給 tanh）。"""
    if isinstance(init, (int, float)):
        return float(init)
    if init == "he":
        return np.sqrt(2 / fan_in)
    if init == "xavier":
        return np.sqrt(2 / (fan_in + fan_out))
    raise ValueError(init)


class Layer:
    """所有層的共同介面。沒有參數的層，params／grads 是空 dict。"""

    def __init__(self) -> None:
        self.params: dict[str, np.ndarray] = {}
        self.grads: dict[str, np.ndarray] = {}

    def forward(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def backward(self, dout: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class Linear(Layer):
    """全連接層：out = X @ W + b。X (n, d_in)、W (d_in, d_out)、b (d_out,)。"""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        init: InitName | float = "he",
        rng: np.random.Generator | None = None,
        dtype: type = np.float64,
    ) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, init_std(init, d_in, d_out), size=(d_in, d_out))
        self.params = {"W": W.astype(dtype), "b": np.zeros(d_out, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return X @ self.params["W"] + self.params["b"]

    def backward(self, dout: np.ndarray) -> np.ndarray:
        self.grads["W"] = self.X.T @ dout
        self.grads["b"] = dout.sum(axis=0)
        return dout @ self.params["W"].T


class ReLU(Layer):
    """out = max(0, X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * self.mask


class Tanh(Layer):
    """out = tanh(X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.out = np.tanh(X)
        return self.out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * (1 - self.out**2)


class SoftmaxCrossEntropy:
    """softmax + 平均交叉熵。forward(Z, y) 回傳純量 loss；backward() 回傳 ∂L/∂Z = (P − Y)/n。"""

    def forward(self, Z: np.ndarray, y: np.ndarray) -> float:
        log_P = Z - logsumexp(Z, axis=1, keepdims=True)
        self.P = np.exp(log_P)
        self.y = y
        return float(-log_P[np.arange(len(y)), y].mean())

    def backward(self) -> np.ndarray:
        n = len(self.y)
        dZ = self.P.copy()
        dZ[np.arange(n), self.y] -= 1
        return dZ / n


def numerical_grad(f, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """中央差分：回傳與 x 同 shape 的 ∂f/∂x。會就地修改 x 再改回來。"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps
        f_plus = f()
        x[i] = old - eps
        f_minus = f()
        x[i] = old
        grad[i] = (f_plus - f_minus) / (2 * eps)
    return grad


def rel_error(a: np.ndarray, b: np.ndarray) -> float:
    """相對誤差 ‖a − b‖ / ‖a + b‖（兩者都是 0 時回傳 0）。"""
    denom = np.linalg.norm(a + b)
    return 0.0 if denom == 0 else float(np.linalg.norm(a - b) / denom)


def check_layer(layer: Layer, X: np.ndarray, seed: int = 0) -> dict[str, float]:
    """單一層的梯度檢查（float64）。f(X) = Σ forward(X) ⊙ G，G 是隨機的上游梯度。

    回傳 {"dX": 相對誤差, "dW": ..., "db": ...}（沒有參數的層只有 dX）。
    """
    G = np.random.default_rng(seed).normal(size=layer.forward(X).shape)
    f = lambda: float(np.sum(layer.forward(X) * G))
    layer.forward(X)
    dX = layer.backward(G)
    errs = {"dX": rel_error(dX, numerical_grad(f, X))}
    for k in layer.params:
        errs["d" + k] = rel_error(layer.grads[k], numerical_grad(f, layer.params[k]))
    return errs

In [ ]:
class Embedding(Layer):
    """查表：out = W[X]。X (n, T) 是整數（0 = padding，那一列固定是 0）、W (V, E)、out (n, T, E)。"""

    def __init__(self, V: int, E: int, std: float = 0.1, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, std, size=(V, E))
        W[0] = 0.0
        self.params = {"W": W}
        self.grads = {"W": np.zeros_like(W)}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return self.params["W"][X]

    def backward(self, dout: np.ndarray) -> None:
        """回傳 None：輸入是整數編號，沒有「對輸入的梯度」。"""
        dW = np.zeros_like(self.params["W"])
        np.add.at(dW, self.X, dout)
        dW[0] = 0.0  # padding 的向量不更新
        self.grads["W"] = dW
        return None


class RNN(Layer):
    """單層 tanh RNN（batch_first）。forward(X, mask) 回傳最後的隱藏狀態 (n, H)。

    forward 存下的 cache：self.X (n, T, E)、self.mask (n, T)、self.Hs (n, T + 1, H)。
    backward 另外存 self.dh_norms (n, T)：‖∂L/∂h‖（讀完第 t 個字母後的隱藏狀態），實驗用。
    """

    def __init__(self, E: int, H: int, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, H)),
            "b": np.zeros(H),
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        Wx, Wh, b = self.params["Wx"], self.params["Wh"], self.params["b"]
        Hs = np.zeros((n, T + 1, Wh.shape[0]))
        for t in range(T):
            h_new = np.tanh(X[:, t] @ Wx + Hs[:, t] @ Wh + b)
            m = mask[:, t, None]
            Hs[:, t + 1] = m * h_new + (1 - m) * Hs[:, t]
        self.X, self.mask, self.Hs = X, mask, Hs
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，並把參數的梯度存進 self.grads。"""
        X, mask, Hs = self.X, self.mask, self.Hs
        n, T, _ = X.shape
        Wx, Wh = self.params["Wx"], self.params["Wh"]
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        dX = np.zeros_like(X)
        self.dh_norms = np.zeros((n, T))
        dh = dh_last
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t] = np.linalg.norm(dh, axis=1)
            m = mask[:, t, None]
            h, h_prev = Hs[:, t + 1], Hs[:, t]
            da = dh * m * (1 - h**2)
            self.grads["Wx"] += X[:, t].T @ da
            self.grads["Wh"] += h_prev.T @ da
            self.grads["b"] += da.sum(axis=0)
            dX[:, t] = da @ Wx.T
            dh = da @ Wh.T + dh * (1 - m)
        return dX


def clip_grad_norm(grads: list[np.ndarray], max_norm: float) -> float:
    """就地把梯度縮放到全域範數 ≤ max_norm。回傳縮放前的全域範數。"""
    total = float(np.sqrt(sum((g**2).sum() for g in grads)))
    scale = 1.0 if total == 0 else min(1.0, max_norm / total)
    for g in grads:
        g *= scale
    return total

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))


class LSTM(Layer):
    """單層 LSTM（batch_first）。forward(X, mask) 回傳最後的隱藏狀態 h_T (n, H)。

    參數的四段順序 [i | f | g | o]（跟 PyTorch 一樣）。
    forward 存下的 cache：self.X、self.mask、self.Hs、self.Cs (n, T + 1, H)、self.Gates (n, T, 4H)。
    backward 另外存 self.dh_norms、self.dc_norms (n, T)：‖∂L/∂h_t‖、‖∂L/∂c_t‖，實驗用。
    """

    def __init__(self, E: int, H: int, forget_bias: float = 1.0, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        b = np.zeros(4 * H)
        b[H : 2 * H] = forget_bias
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, 4 * H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, 4 * H)),
            "b": b,
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.H = H

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        H, Wh, b = self.H, self.params["Wh"], self.params["b"]
        XW = X @ self.params["Wx"]  # (n, T, 4H)：所有時間步的 x_t Wx 一次算好
        Hs = np.zeros((n, T + 1, H))
        Cs = np.zeros((n, T + 1, H))
        Gates = np.zeros((n, T, 4 * H))
        for t in range(T):
            z = XW[:, t] + Hs[:, t] @ Wh + b
            i, f, g, o = sigmoid(z[:, :H]), sigmoid(z[:, H : 2 * H]), np.tanh(z[:, 2 * H : 3 * H]), sigmoid(z[:, 3 * H :])
            c = f * Cs[:, t] + i * g
            h = o * np.tanh(c)
            m = mask[:, t, None]
            Cs[:, t + 1] = m * c + (1 - m) * Cs[:, t]
            Hs[:, t + 1] = m * h + (1 - m) * Hs[:, t]
            Gates[:, t] = np.concatenate([i, f, g, o], axis=1)
        self.X, self.mask, self.Hs, self.Cs, self.Gates = X, mask, Hs, Cs, Gates
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，參數的梯度存進 self.grads。"""
        X, mask, Hs, Cs, Gates = self.X, self.mask, self.Hs, self.Cs, self.Gates
        n, T, _ = X.shape
        H = self.H
        dZ = np.zeros((n, T, 4 * H))
        dh, dc = dh_last, np.zeros_like(dh_last)
        dWh = np.zeros_like(self.params["Wh"])
        self.dh_norms, self.dc_norms = np.zeros((n, T)), np.zeros((n, T))
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t], self.dc_norms[:, t] = np.linalg.norm(dh, axis=1), np.linalg.norm(dc, axis=1)
            m = mask[:, t, None]
            i, f, g, o = Gates[:, t, :H], Gates[:, t, H : 2 * H], Gates[:, t, 2 * H : 3 * H], Gates[:, t, 3 * H :]
            c_prev = Cs[:, t]
            tanh_c = np.tanh(f * c_prev + i * g)  # 這一步「真的」c 的 tanh（padding 的列不會用到）
            dh_m, dc_m = dh * m, dc * m
            dc_tot = dc_m + dh_m * o * (1 - tanh_c**2)
            dz = np.concatenate([dc_tot * g * i * (1 - i), dc_tot * c_prev * f * (1 - f), dc_tot * i * (1 - g**2), dh_m * tanh_c * o * (1 - o)], axis=1)
            dZ[:, t] = dz
            dWh += Hs[:, t].T @ dz
            dh = dz @ self.params["Wh"].T + dh * (1 - m)
            dc = dc_tot * f + dc * (1 - m)
        # Wx、b 的梯度：所有時間步一起算（跟 forward 的 XW 一樣，一次大的矩陣乘法）
        self.grads = {"Wx": np.einsum("nte,ntk->ek", X, dZ), "Wh": dWh, "b": dZ.sum(axis=(0, 1))}
        return dZ @ self.params["Wx"].T

### 3A. `GRU` 的 forward

每一步（PyTorch／cuDNN 的版本）：

$$
\begin{aligned}
a_t &= x_t W_x + b \quad (n, 3H)\text{，三段 } [r \mid z \mid n]；\qquad u_t = h_{t-1} W_h \quad (n, 3H) \\
r_t &= \sigma(a^{(r)}_t + u^{(r)}_t) && \text{重置閘門} \\
z_t &= \sigma(a^{(z)}_t + u^{(z)}_t) && \text{更新閘門} \\
\tilde u_t &= u^{(n)}_t + b_{hn} && \text{「舊的 } h \text{」對候選內容的貢獻} \\
n_t &= \tanh(a^{(n)}_t + r_t \odot \tilde u_t) && \text{候選內容：重置閘門決定要不要參考舊的 } h \\
h_t &= z_t \odot h_{t-1} + (1 - z_t) \odot n_t && \text{插值（前置複習 1）}
\end{aligned}
$$

- $z_t \to 1$：$h_t \approx h_{t-1}$（**直接複製**舊的狀態 → 梯度的捷徑，像 LSTM 的 $f \to 1$）；$z_t \to 0$：整個換成新的 $n_t$。
- $r_t \to 0$：候選內容只看這個字、不看過去（「重新開始」）。
- **跟原論文的差異**：Cho et al. (2014) 是 $n_t = \tanh(x_t W^{(n)}_x + (r_t \odot h_{t-1}) W^{(n)}_h)$ —— 先重置、再乘矩陣；PyTorch 是**先乘矩陣、再重置**（$r_t \odot (h_{t-1}W^{(n)}_h + b_{hn})$），這樣三段的 $h_{t-1}W_h$ 可以一次算完。因為 $r_t$ 乘在 $b_{hn}$ 外面，$b_{hn}$ 不能跟 $b$ 合併 → 多一個偏差參數。本章用 PyTorch 的版本（第 4 節要逐位比對）。
- 也有文獻把 $z$ 的角色反過來寫（$h_t = (1 - z)h_{t-1} + z\,n_t$），只是命名不同。

**mask**：同第 2、3 課，padding 的位置 $h$ 保持不變。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $XW_x + b$ | 所有時間步的輸入側一次算好 | $(n, T, 3H)$ | `XW` |
| $u_t$ | $h_{t-1}W_h$ | $(n, 3H)$ | `hw` |
| $r, z, n$ | 閘門與候選內容 | 各 $(n, H)$ | `r`、`z`、`n_` |
| $\tilde u_t$ | $u^{(n)}_t + b_{hn}$（反向傳播要用） | $(n, H)$ | `hn` |
| Hs | $h_0..h_T$ | $(n, T + 1, H)$ | `self.Hs` |
| Gates | 每一步的 $[r, z, n, \tilde u]$ | $(n, T, 4H)$ | `self.Gates` |

**下標公式**：
- TODO 3：`r = sigmoid(XW[:, t, :H] + hw[:, :H])`、`z = sigmoid(XW[:, t, H:2*H] + hw[:, H:2*H])`。
- TODO 4：`hn = hw[:, 2*H:] + bhn`；`n_ = np.tanh(XW[:, t, 2*H:] + r * hn)`；`h = z * h_prev + (1 - z) * n_`；再用 mask 決定 `Hs[:, t + 1]`。

#### 最小例子：TODO 3、4（$E = H = 1$ 的 GRU，兩步）

$x = (1, 1)$、$h_0 = 0$；$W_x = [0.5, 1.0, 2.0]$（$r, z, n$）、$W_h = [0, 0, 1]$（只有候選內容會看舊的 $h$）、$b = 0$、$b_{hn} = 0$。

1. **正確答案**（純量）：
   - 第 1 步：$r = \sigma(0.5) = 0.6225$、$z = \sigma(1) = 0.7311$、$\tilde u = 1 \cdot h_0 = 0$、$n = \tanh(2) = 0.9640$；$h_1 = 0.7311 \times 0 + 0.2689 \times 0.9640 = 0.2593$。
   - 第 2 步：$\tilde u = h_1 = 0.2593$、$n = \tanh(2 + 0.6225 \times 0.2593) = \tanh(2.1614) = 0.9738$；$h_2 = 0.7311 \times 0.2593 + 0.2689 \times 0.9738 = 0.4514$。
2. **積木**：第 1 步的矩陣版、三段切片。
3. **留給你**：寫進 `GRU.forward`。

In [ ]:
# 最小例子：TODO 3、4（E = H = 1 的 GRU，兩步）
# ---- 1. 正確答案：純量 ----
h_s = 0.0
for step in range(2):
    r_s, z_s = sigmoid(0.5), sigmoid(1.0)
    n_s = np.tanh(2.0 + r_s * (1.0 * h_s))
    h_s = z_s * h_s + (1 - z_s) * n_s
    print(f"第 {step + 1} 步：r = {r_s:.4f}、z = {z_s:.4f}、n = {n_s:.4f}、h = {h_s:.4f}")

# ---- 2. 積木：第 1 步的矩陣版 ----
Xg_mini = np.array([[[1.0], [1.0]]])  # (n=1, T=2, E=1)
Wx_g, Wh_g = np.array([[0.5, 1.0, 2.0]]), np.array([[0.0, 0.0, 1.0]])
XW_mini = Xg_mini @ Wx_g  # (1, 2, 3)
hw_mini = np.zeros((1, 1)) @ Wh_g
H_ = 1
print("\nXW[:, 0] =", XW_mini[:, 0], "；hw =", hw_mini, "；r 段 =", XW_mini[:, 0, :H_] + hw_mini[:, :H_], "；z 段 =", XW_mini[:, 0, H_ : 2 * H_] + hw_mini[:, H_ : 2 * H_])

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 [0. 0.2593 0.4514]）：
# gru_mini = GRU(1, 1); gru_mini.params.update(Wx=Wx_g, Wh=Wh_g, b=np.zeros(3), bhn=np.zeros(1)); gru_mini.forward(Xg_mini, np.ones((1, 2))); print(gru_mini.Hs.ravel().round(4))

In [ ]:
class GRU(Layer):
    """單層 GRU（PyTorch 版本，batch_first）。forward(X, mask) 回傳 h_T (n, H)。

    參數三段的順序 [r | z | n]（跟 PyTorch 一樣）；b 是輸入側的偏差 (3H,)、bhn 是候選內容「h 那一側」的偏差 (H,)。
    cache：self.X、self.mask、self.Hs (n, T + 1, H)、self.Gates (n, T, 4H) = [r, z, n, ũ]。
    backward 另外存 self.dh_norms (n, T)。
    """

    def __init__(self, E: int, H: int, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, 3 * H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, 3 * H)),
            "b": np.zeros(3 * H),
            "bhn": np.zeros(H),
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.H = H

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        H, Wh, bhn = self.H, self.params["Wh"], self.params["bhn"]
        XW = X @ self.params["Wx"] + self.params["b"]  # (n, T, 3H)
        Hs = np.zeros((n, T + 1, H))
        Gates = np.zeros((n, T, 4 * H))
        for t in range(T):
            h_prev = Hs[:, t]
            hw = h_prev @ Wh  # (n, 3H)
            # TODO 3（程式）：r、z（見下標公式）
            raise NotImplementedError("TODO 3：重置、更新閘門")
            # TODO 4（程式）：hn、n_、h，再用 mask 決定 Hs[:, t + 1]
            raise NotImplementedError("TODO 4：候選內容與插值")
            Gates[:, t] = np.concatenate([r, z, n_, hn], axis=1)
        self.X, self.mask, self.Hs, self.Gates = X, mask, Hs, Gates
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，參數的梯度存進 self.grads。"""
        X, mask, Hs, Gates = self.X, self.mask, self.Hs, self.Gates
        n, T, _ = X.shape
        H, Wh = self.H, self.params["Wh"]
        dZx = np.zeros((n, T, 3 * H))  # ∂L/∂(輸入側的三段)：給 Wx、b
        dWh, dbhn = np.zeros_like(Wh), np.zeros(H)
        dh = dh_last
        self.dh_norms = np.zeros((n, T))
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t] = np.linalg.norm(dh, axis=1)
            m = mask[:, t, None]
            r, z, n_, hn = Gates[:, t, :H], Gates[:, t, H : 2 * H], Gates[:, t, 2 * H : 3 * H], Gates[:, t, 3 * H :]
            h_prev = Hs[:, t]
            # TODO 6（程式）：GRU 的 BPTT 一步（見 3B 的表格）
            #   - dh_m = dh * m
            #   - dn = dh_m * (1 − z)；dz = dh_m * (h_prev − n_)        ← 前置複習 2
            #   - da_n = dn * (1 − n_²)；dr = da_n * hn；dhn = da_n * r
            #   - da_r = dr * r * (1 − r)；da_z = dz * z * (1 − z)
            #   - dZx[:, t] = [da_r, da_z, da_n]（輸入側）；dzh = [da_r, da_z, dhn]（h 側 —— 第三段不一樣！）
            #   - dWh += h_prev.T @ dzh；dbhn += dhn.sum(axis=0)
            #   - dh = dzh @ Wh.T + dh_m * z + dh * (1 − m)        ← 三條路：經過 Wh、直接複製（z）、padding
            raise NotImplementedError("TODO 6：GRU 的 BPTT")
        self.grads = {"Wx": np.einsum("nte,ntk->ek", X, dZx), "Wh": dWh, "b": dZx.sum(axis=(0, 1)), "bhn": dbhn}
        return dZx @ self.params["Wx"].T

### 3B. GRU 的反向傳播

從 $\delta_t = \partial L / \partial h_t$ 出發，每往回一步（先假設不是 padding）：

| 量 | 公式 | 怎麼來的 |
| --- | --- | --- |
| $\partial L/\partial n_t$ | $\delta_t \odot (1 - z_t)$ | 插值對 $n$ 的導數（前置複習 2） |
| $\partial L/\partial z_t$ | $\delta_t \odot (h_{t-1} - n_t)$ | 插值對 $z$ 的導數 |
| $\partial L/\partial (\text{tanh 之前})$ | $\partial L/\partial n_t \odot (1 - n_t^2)$ | |
| $\partial L/\partial r_t$ | 上一列 $\odot\ \tilde u_t$ | $n = \tanh(\cdots + r \odot \tilde u)$ |
| $\partial L/\partial \tilde u_t$ | 上一列（tanh 之前） $\odot\ r_t$ | 同上 → 給 $W_h$ 的第三段與 $b_{hn}$ |
| 閘門的「sigmoid 之前」 | $\partial L/\partial r \odot r(1 - r)$、$\partial L/\partial z \odot z(1 - z)$ | |
| $\partial L/\partial W_x$、$b$ | 輸入側三段 $[da_r, da_z, da_n]$，同 RNN、LSTM | 每一步加總 |
| $\partial L/\partial W_h$ | $h_{t-1}^\top [da_r, da_z, \partial L/\partial\tilde u]$ | **第三段**用的是 $\partial L/\partial\tilde u$（乘過 $r$），不是 $da_n$ |
| $\delta_{t-1}$ | $[da_r, da_z, \partial L/\partial\tilde u]\,W_h^\top + \delta_t \odot z_t$ | 兩條路：經過三個閘門的 $W_h$，以及插值裡**直接複製**的 $z_t \odot h_{t-1}$ |

最後一行的第二項 $\delta_t \odot z_t$ 是 GRU 的「捷徑」：如果 $z_t \approx 1$，梯度幾乎原封不動地傳到 $h_{t-1}$ —— 跟 LSTM 的 $\delta^c_{t-1} = \partial L/\partial c_t \odot f_t$ 一樣的角色，只是 GRU 把它放在 $h$ 上。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\delta_t$ | $\partial L/\partial h_t$ | $(n, H)$ | `dh` |
| $da_r, da_z, da_n$ | 三段「啟動函數之前」的梯度 | 各 $(n, H)$ | `da_r`、`da_z`、`da_n` |
| $\partial L/\partial\tilde u$ | 候選內容 $h$ 側的梯度 | $(n, H)$ | `dhn` |
| dZx | 輸入側三段（全部時間步） | $(n, T, 3H)$ | `dZx` |

In [ ]:
# TODO 5（問答）推導與比較（先在紙上，再寫 TODO 6）
#
# (1) 用 TODO 3 最小例子（兩步、loss L = h_2）：
#     (a) ∂L/∂z_2 = h_1 − n_2 = ?（用最小例子的數字）它是正還是負？「把 z 調大」會讓 h_2 變大還是變小？
#     (b) ∂L/∂h_1 的「直接複製」那一項 = z_2 = ?；經過 Wh 的那一項只有候選內容那段（因為 Wh 的 r、z 段是 0）= ?
#     (c) 下一格的最小例子用數值微分驗證四個數字。
# 答：
#
# (2) 跟 LSTM 比：
#     (a) LSTM 往回傳的捷徑是 δ^c ⊙ f；GRU 是 δ ⊙ z。兩者都是「逐元素乘一個 0～1 的閘門」。差別在哪一個狀態上？
#     (b) GRU 少了輸出閘門：LSTM 可以把記憶 c「藏起來」不輸出（o ≈ 0），GRU 能嗎？這對什麼樣的任務可能有影響？
#     (c) GRU 的 z 同時控制「留舊的」和「寫新的」（加起來 = 1）。LSTM 的 f、i 可以「兩個都 ≈ 1」（一邊記住、一邊累加）。舉一個需要「累加」的例子（提示：數數）。
# 答：

#### 最小例子：TODO 6（GRU 的 BPTT）

同 TODO 3 的最小例子，loss $= h_2$。對 $W_x$ 的三個數字（$r, z, n$ 段）與 $W_h$ 的第三段（候選內容）各做數值微分。

1. **正確答案**：應為 $\partial L/\partial W_x \approx (0.0008, -0.2807, 0.0280)$、$\partial L/\partial W_h^{(n)} \approx 0.0022$ —— $z$ 段是**負的**（TODO 5 (1)(a)）。
2. **積木**：第 2 步的 $\partial L/\partial z_2 = h_1 - n_2$。
3. **留給你**：TODO 6，對照 1。

In [ ]:
# 最小例子：TODO 6（GRU 的 BPTT）
def loss_two_steps(w: np.ndarray) -> float:
    """w = (Wx_r, Wx_z, Wx_n, Wh_n)；其他同上。"""
    h = 0.0
    for x in [1.0, 1.0]:
        r_, z_ = sigmoid(w[0] * x), sigmoid(w[1] * x)
        n__ = np.tanh(w[2] * x + r_ * (w[3] * h))
        h = z_ * h + (1 - z_) * n__
    return h


# ---- 1. 正確答案：數值微分 ----
w0 = np.array([0.5, 1.0, 2.0, 1.0])
num = np.array([(loss_two_steps(w0 + e) - loss_two_steps(w0 - e)) / 2e-6 for e in np.eye(4) * 1e-6])
print("數值微分 (Wx_r, Wx_z, Wx_n, Wh_n)：", num.round(4))

# ---- 2. 積木 ----
h1_s = (1 - sigmoid(1.0)) * np.tanh(2.0)
n2_s = np.tanh(2.0 + sigmoid(0.5) * h1_s)
print(f"∂L/∂z_2 = h_1 − n_2 = {h1_s:.4f} − {n2_s:.4f} = {h1_s - n2_s:.4f}")

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（Wx 應為 [[0.0008 -0.2807 0.028]]、Wh 的第三個應為 0.0022）：
# gru_mini = GRU(1, 1); gru_mini.params.update(Wx=Wx_g.copy(), Wh=Wh_g.copy(), b=np.zeros(3), bhn=np.zeros(1)); gru_mini.forward(Xg_mini, np.ones((1, 2))); gru_mini.backward(np.ones((1, 1))); print(gru_mini.grads["Wx"].round(4), gru_mini.grads["Wh"].round(4))

### 3C. 梯度檢查、在子集上訓練 RNN／LSTM／GRU

`TextRNNNumpy` 跟第 3 課一樣（Embedding → 循環層 → Linear → softmax、Adam、梯度裁剪、`restore_best`），只是 `cell` 多了 `"gru"`。

**子集**：同第 3 課 —— train 前 8,000 篇、validation 前 1,000 篇，每篇最後 100 個字，$E = 32$、$H = 64$，Adam lr = 2e-3，6 epochs。三種循環層用同樣的設定與 seed。

In [ ]:
def lengths_to_mask(lengths: np.ndarray, T: int) -> np.ndarray:
    return (np.arange(T)[None, :] < lengths[:, None]).astype(float)


class TextRNNNumpy:
    """Embedding → RNN／LSTM／GRU → Linear → softmax（2 類）。Adam + 梯度裁剪（同第 3 課）。"""

    def __init__(self, V: int, cell: str = "gru", E: int = 32, H: int = 64, lr: float = 2e-3, batch_size: int = 64,
                 n_epochs: int = 6, clip: float = 5.0, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.emb = Embedding(V, E, rng=rng)
        self.rnn = {"rnn": lambda: RNN(E, H, rng=rng), "lstm": lambda: LSTM(E, H, rng=rng), "gru": lambda: GRU(E, H, rng=rng)}[cell]()
        self.fc = Linear(H, 2, init="xavier", rng=rng)
        self.loss = SoftmaxCrossEntropy()
        self.layers = [self.emb, self.rnn, self.fc]
        self.cell, self.lr, self.batch_size, self.n_epochs, self.clip, self.seed = cell, lr, batch_size, n_epochs, clip, seed

    def _param_list(self) -> list[tuple[dict, dict, str]]:
        return [(layer.params, layer.grads, k) for layer in self.layers for k in layer.params]

    def _adam_step(self, beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8) -> None:
        if not hasattr(self, "_adam_t"):
            self._adam_t = 0
            self._adam_m = [np.zeros_like(p[k]) for p, _, k in self._param_list()]
            self._adam_v = [np.zeros_like(p[k]) for p, _, k in self._param_list()]
        self._adam_t += 1
        for (p, g, k), m, v in zip(self._param_list(), self._adam_m, self._adam_v):
            m *= beta1
            m += (1 - beta1) * g[k]
            v *= beta2
            v += (1 - beta2) * g[k] ** 2
            p[k] -= self.lr * (m / (1 - beta1**self._adam_t)) / (np.sqrt(v / (1 - beta2**self._adam_t)) + eps)

    def forward(self, Xi: np.ndarray, lengths: np.ndarray) -> np.ndarray:
        T = int(lengths.max())
        return self.fc.forward(self.rnn.forward(self.emb.forward(Xi[:, :T]), lengths_to_mask(lengths, T)))

    def loss_and_backward(self, Xi: np.ndarray, lengths: np.ndarray, y: np.ndarray) -> float:
        loss = self.loss.forward(self.forward(Xi, lengths), y)
        self.emb.backward(self.rnn.backward(self.fc.backward(self.loss.backward())))
        return loss

    def predict_proba(self, Xi: np.ndarray, lengths: np.ndarray, batch_size: int = 500) -> np.ndarray:
        out = []
        for s in range(0, len(Xi), batch_size):
            Z = self.forward(Xi[s : s + batch_size], lengths[s : s + batch_size])
            P = np.exp(Z - Z.max(axis=1, keepdims=True))
            out.append(P / P.sum(axis=1, keepdims=True))
        return np.vstack(out)

    def fit(self, Xi, lengths, y, Xi_val=None, len_val=None, y_val=None, verbose: bool = True) -> "TextRNNNumpy":
        rng = np.random.default_rng(self.seed + 1)
        order = np.argsort(lengths, kind="stable")
        batches = [order[s : s + self.batch_size] for s in range(0, len(order), self.batch_size)]
        self.history_ = {"train_loss": [], "val_loss": [], "val_acc": [], "seconds": []}
        best_loss, best_state = np.inf, None
        for epoch in range(self.n_epochs):
            t0, total = time.perf_counter(), 0.0
            for b in rng.permutation(len(batches)):
                idx = batches[b]
                total += self.loss_and_backward(Xi[idx], lengths[idx], y[idx]) * len(idx)
                clip_grad_norm([g[k] for _, g, k in self._param_list()], self.clip)
                self._adam_step()
            self.history_["train_loss"].append(total / len(y))
            msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
            if Xi_val is not None:
                s_ = val_scores(self.predict_proba(Xi_val, len_val), y_val)
                self.history_["val_loss"].append(s_["log_loss"])
                self.history_["val_acc"].append(s_["accuracy"])
                msg += f"  val_loss {s_['log_loss']:.4f}  val_acc {s_['accuracy']:.4f}"
                if s_["log_loss"] < best_loss:
                    best_loss, self.best_epoch_ = s_["log_loss"], epoch + 1
                    best_state = [(p, k, p[k].copy()) for p, _, k in self._param_list()]
            self.history_["seconds"].append(time.perf_counter() - t0)
            if verbose:
                print(msg + f"  ({self.history_['seconds'][-1]:.1f}s)")
        if best_state is not None:
            for p, k, v in best_state:
                p[k] = v
        return self

In [ ]:
# 梯度檢查（V = 8、E = 3、H = 4，含 padding）
gc = TextRNNNumpy(V=8, E=3, H=4, seed=1)
with np.errstate(all="ignore"):
    gc.rnn.params["b"] = np.random.default_rng(2).normal(0, 0.3, size=gc.rnn.params["b"].shape)  # 偏差也給非 0 的值，才檢查得到
    gc.rnn.params["bhn"] = np.random.default_rng(3).normal(0, 0.3, size=gc.rnn.params["bhn"].shape)
Xi_gc = np.array([[2, 3, 4, 5, 6], [7, 1, 2, 0, 0], [3, 3, 0, 0, 0]])
len_gc = np.array([5, 3, 2])
y_gc = np.array([0, 1, 1])
gc.loss_and_backward(Xi_gc, len_gc, y_gc)
f_gc = lambda: gc.loss.forward(gc.forward(Xi_gc, len_gc), y_gc)
for layer, name in [(gc.emb, "emb"), (gc.rnn, "gru"), (gc.fc, "fc")]:
    for k in layer.params:
        numeric = numerical_grad(f_gc, layer.params[k])
        if name == "emb":
            numeric[0] = 0.0
        print(f"✅ {name}.{k:3s} 相對誤差 {rel_error(layer.grads[k], numeric):.1e}（應 < 1e-6）")
print("✅ padding 不改變結果？", np.allclose(gc.forward(Xi_gc[1:2, :3], len_gc[1:2]), gc.forward(Xi_gc, len_gc)[1:2]))

In [ ]:
# 子集：RNN／LSTM／GRU 各 6 epochs（這台電腦：RNN 一個 epoch 約 5 秒、LSTM 約 18 秒、GRU 約 15 秒）
SUB_TR, SUB_VA, SUB_LEN = 8_000, 1_000, 100
Xs_tr, ls_tr = encode(tok_tr[:SUB_TR], max_len=SUB_LEN)
Xs_va, ls_va = encode(tok_va[:SUB_VA], max_len=SUB_LEN)
ys_tr, ys_va = y_tr[:SUB_TR], y_va[:SUB_VA]
np_models = {}
for cell in ["rnn", "lstm", "gru"]:
    print(f"── {cell} ──")
    np_models[cell] = TextRNNNumpy(V, cell=cell).fit(Xs_tr, ls_tr, ys_tr, Xs_va, ls_va, ys_va)
sub_table = pd.DataFrame({
    cell: {**val_scores(m.predict_proba(Xs_va, ls_va), ys_va), "recurrent params": sum(v.size for v in m.rnn.params.values()),
           "sec/epoch": np.mean(m.history_["seconds"]), "best_epoch": m.best_epoch_}
    for cell, m in np_models.items()
}).T
display(sub_table)

In [ ]:
# TODO 7（問答）
#
# (1) 梯度檢查：每個參數的相對誤差都 < 1e-6 嗎？為什麼要把 b、bhn 先設成非 0 的值再檢查？（提示：bhn 被 r 乘；全是 0 的時候哪些項會消失？）
# (2) 子集上的三個模型：
#     (a) validation accuracy、循環層的參數量、每個 epoch 的時間各是多少？GRU 的參數量是 LSTM 的幾分之幾？
#     (b) GRU 和 LSTM 的差距，跟第 3 課觀察到的 seed 雜訊（約 0.02）比起來大嗎？
# 答：

### 3D. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 梯度能傳多遠：RNN、LSTM、GRU 的 $\lVert\partial L/\partial h_t\rVert$（LSTM 另看 $\lVert\partial L/\partial c_t\rVert$）隨離結尾的距離怎麼變？ |
| 2 | 更新閘門 $z$ 在做什麼：值的分布；哪些字讓 GRU「保留舊狀態」（$z$ 大）、哪些字讓它「寫入新內容」（$z$ 小）？ |
| 3 | 逐字的預測：GRU 和 LSTM 在同樣的句子上，什麼時候改變心意？ |

In [ ]:
# 實驗 1：梯度能傳多遠（validation 子集裡長度 = 100 的影評）
full_idx = np.flatnonzero(ls_va == SUB_LEN)[:200]
fig, ax = plt.subplots(figsize=(9, 4))
for cell, m in np_models.items():
    m.loss_and_backward(Xs_va[full_idx], ls_va[full_idx], ys_va[full_idx])
    dh = m.rnn.dh_norms.mean(axis=0)[::-1]
    ax.semilogy(dh / dh[0], label=f"{cell}: ‖∂L/∂h‖")
    msg = f"{cell:4s}：離結尾 25、50、99 步 ‖∂L/∂h‖ 剩 {[f'{v:.1e}' for v in (dh / dh[0])[[25, 50, 99]]]}"
    if cell == "lstm":
        dc = m.rnn.dc_norms.mean(axis=0)[::-1]
        ax.semilogy(dc / dh[0], "--", label="lstm: ‖∂L/∂c‖")
        msg += f"；‖∂L/∂c‖ 剩 {[f'{v:.1e}' for v in (dc / dh[0])[[25, 50, 99]]]}"
    print(msg)
ax.set_xlabel("steps before the last token")
ax.set_ylabel("gradient norm (relative)")
ax.legend()
plt.show()

In [ ]:
# 實驗 2：更新閘門 z（訓練好的 GRU）
gru_np = np_models["gru"]
gru_np.forward(Xs_va[full_idx], ls_va[full_idx])
H_np = gru_np.rnn.H
Z = gru_np.rnn.Gates[:, :, H_np : 2 * H_np]  # (200, 100, H)
R = gru_np.rnn.Gates[:, :, :H_np]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(Z.ravel(), bins=50, alpha=0.6, label="update gate z")
axes[0].hist(R.ravel(), bins=50, alpha=0.6, label="reset gate r")
axes[0].legend()
unit_z = Z.mean(axis=(0, 1))
axes[1].bar(np.arange(H_np), np.sort(unit_z)[::-1])
axes[1].set_title("mean update gate per hidden unit (sorted)")
plt.show()
print(f"z > 0.9 的比例：{np.mean(Z > 0.9):.1%}；z < 0.1：{np.mean(Z < 0.1):.1%}；每個單元平均 z 的範圍：{unit_z.min():.3f}～{unit_z.max():.3f}")
lstm_np = np_models["lstm"]
lstm_np.forward(Xs_va[full_idx], ls_va[full_idx])
F_l = lstm_np.rnn.Gates[:, :, lstm_np.rnn.H : 2 * lstm_np.rnn.H]
print(f"對照 LSTM 的遺忘閘門 f：平均 {F_l.mean():.3f}、f > 0.9 的比例 {np.mean(F_l > 0.9):.1%}")

word_ids = Xs_va[full_idx][:, :SUB_LEN].ravel()
mean_z = pd.Series(Z.mean(axis=2).ravel()).groupby(word_ids).agg(["mean", "size"])
mean_z = mean_z[mean_z["size"] >= 20].sort_values("mean")
print("z 最小（寫入新內容）的字：", [ID_TO_WORD[k] for k in mean_z.index[:12]])
print("z 最大（保留舊狀態）的字：", [ID_TO_WORD[k] for k in mean_z.index[-12:]])

In [ ]:
# 實驗 3：逐字的預測（GRU vs LSTM，NumPy 子集模型）
def prefix_pos_prob(model: TextRNNNumpy, tokens: list[str]) -> np.ndarray:
    Xi, L = encode([tokens], max_len=len(tokens))
    model.forward(Xi, L)
    Zl = model.rnn.Hs[0, 1:] @ model.fc.params["W"] + model.fc.params["b"]
    P = np.exp(Zl - Zl.max(axis=1, keepdims=True))
    return (P / P.sum(axis=1, keepdims=True))[:, 1]


demo = [
    "i expected a great movie , but it was boring and the acting was terrible .",
    "the first half was terrible , but the ending was brilliant and i loved it .",
    "not bad at all . i was surprised .",
]
fig, axes = plt.subplots(len(demo), 1, figsize=(15, 2.5 * len(demo)))
for ax, sent in zip(axes, demo):
    toks = sent.split()
    for cell in ["lstm", "gru"]:
        ax.plot(prefix_pos_prob(np_models[cell], toks), "o-", label=cell)
    ax.axhline(0.5, color="gray", ls=":")
    ax.set_xticks(np.arange(len(toks)), toks, rotation=30, fontsize=9)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# TODO 8（問答）實驗 1–3
#
# (1) 實驗 1：離結尾 50、99 步時，三個模型的梯度各剩多少？GRU 的曲線比較像 RNN 還是 LSTM？用 3B 表格最後一行（δ ⊙ z 那一項）解釋。
# 答：
#
# (2) 實驗 2：
#     (a) z 的值大多在哪裡？有沒有「幾乎都在複製」（平均 z 很大）的單元？跟 LSTM 的 f 比呢？
#     (b) z 最小（寫入新內容）的字是哪些？跟情緒有關嗎？z 最大的呢？
# 答：
#
# (3) 實驗 3：在「but」之後，GRU 和 LSTM 誰比較快改變心意？兩個模型的曲線形狀像嗎？
# 答：

---
## 4. 套件實作：PyTorch `nn.GRU`

| PyTorch | 對應手寫的 | 注意 |
| --- | --- | --- |
| `nn.GRU(E, H, batch_first=True)` | `GRU` | `weight_ih_l0` = $W_x^\top$ $(3H, E)$、`weight_hh_l0` = $W_h^\top$；三段順序 **$r, z, n$** |
| `bias_ih_l0`（$3H$）、`bias_hh_l0`（$3H$） | `b`、`bhn` | $r$、$z$ 段兩個偏差相加 = 我們的 `b`；$n$ 段：`bias_ih` = `b` 的第三段、`bias_hh` 的第三段 = `bhn`（被 $r$ 乘） |
| 回傳 `output, h_n` | `Hs` | 跟 RNN 一樣只有 $h_n$（沒有 $c_n$） |

**本機用 CPU**：同第 2、3 課。

In [ ]:
import torch
from torch import nn
from learning_map.utils import count_parameters, get_device, set_seed

device = torch.device("cpu")


class TextRNNTorch(nn.Module):
    """Embedding → nn.RNN／nn.LSTM／nn.GRU（可雙向）→ Linear(2)。LSTM 的遺忘閘門偏差設 1（同第 3 課）。"""

    def __init__(self, V: int, cell: str = "gru", E: int = 64, H: int = 64, bidirectional: bool = False) -> None:
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.rnn = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell](E, H, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(H * (2 if bidirectional else 1), 2)
        if cell == "lstm":
            with torch.no_grad():
                for name, p in self.rnn.named_parameters():
                    if name.startswith("bias_ih"):
                        p[H : 2 * H] = 1.0
                    elif name.startswith("bias_hh"):
                        p[H : 2 * H] = 0.0

    def forward(self, X: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(X), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        if isinstance(h_n, tuple):
            h_n = h_n[0]
        h = torch.cat([h_n[-2], h_n[-1]], dim=1) if self.rnn.bidirectional else h_n[-1]
        return self.fc(h)


def load_from_numpy(net: TextRNNTorch, model: TextRNNNumpy) -> TextRNNTorch:
    """把手寫的 GRU 權重複製進 PyTorch（轉置；bhn 放進 bias_hh 的第三段）。"""
    H = model.rnn.H
    with torch.no_grad():
        net.emb.weight.copy_(torch.as_tensor(model.emb.params["W"]))
        net.rnn.weight_ih_l0.copy_(torch.as_tensor(model.rnn.params["Wx"].T))
        net.rnn.weight_hh_l0.copy_(torch.as_tensor(model.rnn.params["Wh"].T))
        net.rnn.bias_ih_l0.copy_(torch.as_tensor(model.rnn.params["b"]))
        net.rnn.bias_hh_l0.zero_()
        net.rnn.bias_hh_l0[2 * H :] = torch.as_tensor(model.rnn.params["bhn"])
        net.fc.weight.copy_(torch.as_tensor(model.fc.params["W"].T))
        net.fc.bias.copy_(torch.as_tensor(model.fc.params["b"]))
    return net


idx_chk = np.arange(16)
loss_np = gru_np.loss_and_backward(Xs_va[idx_chk], ls_va[idx_chk], ys_va[idx_chk])
net_chk = load_from_numpy(TextRNNTorch(V, E=32, H=64).double(), gru_np)
T_chk = int(ls_va[idx_chk].max())
loss_t = nn.functional.cross_entropy(net_chk(torch.as_tensor(Xs_va[idx_chk, :T_chk]), torch.as_tensor(ls_va[idx_chk])), torch.as_tensor(ys_va[idx_chk]))
loss_t.backward()
H64 = gru_np.rnn.H
print(f"✅ loss：手寫 {loss_np:.10f}、PyTorch {loss_t.item():.10f}")
for k, (a, b) in {
    "emb.W": (gru_np.emb.grads["W"], net_chk.emb.weight.grad.numpy()),
    "gru.Wx": (gru_np.rnn.grads["Wx"], net_chk.rnn.weight_ih_l0.grad.numpy().T),
    "gru.Wh": (gru_np.rnn.grads["Wh"], net_chk.rnn.weight_hh_l0.grad.numpy().T),
    "gru.b": (gru_np.rnn.grads["b"], net_chk.rnn.bias_ih_l0.grad.numpy()),
    "gru.bhn": (gru_np.rnn.grads["bhn"], net_chk.rnn.bias_hh_l0.grad.numpy()[2 * H64 :]),
    "fc.W": (gru_np.fc.grads["W"], net_chk.fc.weight.grad.numpy().T),
}.items():
    print(f"✅ {k:8s} 梯度最大差異 {np.abs(a - b).max():.1e}（應 < 1e-10）")

#### 用 PyTorch 訓練：GRU vs LSTM（同樣的 $E = H = 64$、最後 200 個字、5 epochs）

`train_seq` 跟第 3 課相同（Adam 2e-3、梯度裁剪 5、依長度分桶、每個 epoch 量 validation、`restore_best`）。

In [ ]:
@torch.no_grad()
def predict_proba_seq(net: nn.Module, Xi: np.ndarray, lengths: np.ndarray, batch_size: int = 1000) -> np.ndarray:
    net.eval()
    out = []
    for s in range(0, len(Xi), batch_size):
        L = lengths[s : s + batch_size]
        out.append(torch.softmax(net(torch.as_tensor(Xi[s : s + batch_size, : L.max()], device=device), torch.as_tensor(L)).double(), dim=1).cpu().numpy())
    return np.vstack(out)


def train_seq(net: nn.Module, Xi: np.ndarray, lengths: np.ndarray, y: np.ndarray, Xi_val: np.ndarray | None = None,
              len_val: np.ndarray | None = None, y_val: np.ndarray | None = None, *, lr: float = 2e-3, batch_size: int = 64,
              n_epochs: int = 5, clip: float = 5.0, seed: int = 0, restore_best: bool = True, verbose: bool = True) -> dict:
    """同第 3 課：Adam + 梯度裁剪 + 依長度分桶。"""
    set_seed(seed)
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    order = np.argsort(lengths, kind="stable")
    batches = [order[s : s + batch_size] for s in range(0, len(order), batch_size)]
    rng = np.random.default_rng(seed)
    hist: dict = {"train_loss": [], "val_loss": [], "val_acc": [], "seconds": []}
    best_loss, best_epoch, best_state = np.inf, 0, None
    for epoch in range(n_epochs):
        net.train()
        t0, total = time.perf_counter(), 0.0
        for b in rng.permutation(len(batches)):
            idx = batches[b]
            L = lengths[idx]
            loss = loss_fn(net(torch.as_tensor(Xi[idx, : L.max()], device=device), torch.as_tensor(L)), torch.as_tensor(y[idx], device=device))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
            total += loss.item() * len(idx)
        hist["train_loss"].append(total / len(y))
        msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
        if Xi_val is not None:
            s_ = val_scores(predict_proba_seq(net, Xi_val, len_val), y_val)
            hist["val_loss"].append(s_["log_loss"])
            hist["val_acc"].append(s_["accuracy"])
            msg += f"  val_loss {s_['log_loss']:.4f}  val_acc {s_['accuracy']:.4f}"
            if s_["log_loss"] < best_loss:
                best_loss, best_epoch = s_["log_loss"], epoch + 1
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        hist["seconds"].append(time.perf_counter() - t0)
        if verbose:
            print(msg + f"  ({hist['seconds'][-1]:.0f}s)")
    hist["best_epoch"], hist["best_val_loss"] = best_epoch, best_loss
    if restore_best and best_state is not None:
        net.load_state_dict(best_state)
    return hist


torch_models, torch_hists = {}, {}
for cell in ["lstm", "gru"]:
    print(f"── {cell} ──")
    set_seed(0)
    net = TextRNNTorch(V, cell=cell)
    torch_hists[cell] = train_seq(net, Xi_tr, len_tr, y_tr, Xi_va, len_va, y_va)
    torch_models[cell] = net
    val_table.loc[f"{cell.upper()} (PyTorch, E=H=64, seed 0)"] = val_scores(predict_proba_seq(net, Xi_va, len_va))
for cell, net in torch_models.items():
    print(f"{cell}：循環層參數 {count_parameters(net.rnn):,}；全部 {count_parameters(net):,}；每個 epoch {np.mean(torch_hists[cell]['seconds']):.0f} 秒")
display(val_table)

In [ ]:
# TODO 9（問答）PyTorch
#
# (1) 核對：loss 與每個梯度（含 bhn）的差異是多少？如果把 bhn 跟 b 的第三段合併成一個偏差，核對會怎樣？為什麼？
# (2) GRU vs LSTM（seed 0）：validation accuracy、循環層參數量、每個 epoch 的時間各差多少？跟 TODO 2 (1) 的預測比。
#     如果 GRU 參數比較少、卻跑得比較慢：參數量（計算量）只是速度的一部分，另一部分是「實作」——
#     PyTorch 在 CPU 上的 LSTM、GRU 用的是不同的 kernel，最佳化程度不同（GPU 上的 cuDNN 又是另一回事）。速度一定要實測，不能只看參數量。
# 答：

---
## 5. 調參與公平比較

「GRU 比 LSTM 好嗎？」要回答這個問題，得排除兩個干擾：

1. **seed 雜訊**：同一個模型換個 seed，val accuracy 就可能差 0.01～0.02（第 3 課 5C）→ 每個模型跑 **3 個 seed**，看平均與標準差。
2. **參數量**：同樣的 $H$，GRU 的循環層比 LSTM 少約 25% 的參數 → 另外比「**參數量相同**」的 GRU（$H$ 調大到循環層參數 ≈ LSTM $H = 64$）。

| 小節 | 做什麼 |
| --- | --- |
| 5A | GRU 的 $H$：32、64、128 |
| 5B | GRU vs LSTM：同樣 $H = 64$、以及參數量相同的 GRU，各 3 個 seed |

In [ ]:
# 5A. GRU 的 H（seed 0）
rows = []
for H_ in [32, 64, 128]:
    if H_ == 64:
        net, h = torch_models["gru"], torch_hists["gru"]
    else:
        set_seed(0)
        net = TextRNNTorch(V, cell="gru", H=H_)
        h = train_seq(net, Xi_tr, len_tr, y_tr, Xi_va, len_va, y_va, verbose=False)
    rows.append({"H": H_, "recurrent params": count_parameters(net.rnn), **val_scores(predict_proba_seq(net, Xi_va, len_va)),
                 "best_epoch": h["best_epoch"], "sec/epoch": np.mean(h["seconds"])})
display(pd.DataFrame(rows).set_index("H"))

In [ ]:
# 5B. GRU vs LSTM：各 3 個 seed；以及循環層參數量相同的 GRU
def recurrent_params(cell: str, E: int, H: int) -> int:
    k = {"lstm": 4, "gru": 3}[cell]
    return k * H * (E + H) + 2 * k * H  # 權重 + 兩組偏差（PyTorch）


H_eq = min(range(64, 128), key=lambda h: abs(recurrent_params("gru", 64, h) - recurrent_params("lstm", 64, 64)))
print(f"LSTM H=64 的循環層參數 {recurrent_params('lstm', 64, 64):,}；GRU H={H_eq} 是 {recurrent_params('gru', 64, H_eq):,}")
configs = {"LSTM H=64": ("lstm", 64), "GRU H=64": ("gru", 64), f"GRU H={H_eq} (same params)": ("gru", H_eq)}
seed_rows = []
for name, (cell, H_) in configs.items():
    for seed in [0, 1, 2]:
        if seed == 0 and H_ == 64:
            net, h = torch_models[cell], torch_hists[cell]
        else:
            set_seed(seed)
            net = TextRNNTorch(V, cell=cell, H=H_)
            h = train_seq(net, Xi_tr, len_tr, y_tr, Xi_va, len_va, y_va, seed=seed, verbose=False)
        seed_rows.append({"model": name, "seed": seed, **val_scores(predict_proba_seq(net, Xi_va, len_va)), "best_epoch": h["best_epoch"],
                          "sec/epoch": np.mean(h["seconds"])})
seed_tbl = pd.DataFrame(seed_rows)
summary = seed_tbl.groupby("model")[["accuracy", "log_loss", "roc_auc", "best_epoch", "sec/epoch"]].agg(["mean", "std"]).round(4)
display(seed_tbl.pivot(index="seed", columns="model", values="accuracy"))
display(summary)

In [ ]:
# TODO 10（問答）調參與公平比較
#
# (1) 5A：H = 32、64、128 的 accuracy 與時間。值得把 H 調大嗎？
# (2) 5B：
#     (a) 三個設定的平均 ± 標準差各是多少？GRU H=64 和 LSTM H=64 的差距，跟標準差比大還是小？
#     (b) 「參數量相同」的 GRU 跟 LSTM 比呢？
#     (c) 結論：在這份資料上，你能說 GRU 比 LSTM 好（或差）嗎？如果「分不出來」，你會選哪一個？為什麼？（提示：速度、參數量）
#     (d) Chung et al. (2014) 與 Jozefowicz et al. (2015) 的結論大致是「GRU 和 LSTM 互有勝負、差距通常很小」—— 你的結果一致嗎？
# 答：

---
## 6. 測試集評估

依 README：用最佳設定在 train + validation（25,000 篇）refit、epoch 數用 5B 的 best epoch 平均，測試集只評估一次。

**選設定**：5B 裡三個設定的平均 val accuracy 差距在一個標準誤（$\sqrt{0.86 \times 0.14 / 5000} \approx 0.005$）以內時，選**每個 epoch 最快**的（one-standard-error rule，同第 3 課）。

另外在測試集上做一個**配對 bootstrap**（前置複習 5）：GRU 與 LSTM（都在 train 上、seed 0 訓練）在同樣 25,000 篇上的 accuracy 差距，95% 信賴區間是多少？

In [ ]:
from learning_map.metrics import classification_metrics
from learning_map.results import compare, log_result

BENCH = "imdb"
means = summary[("accuracy", "mean")]
SE = float(np.sqrt(0.86 * 0.14 / len(y_va)))
within = [m for m in means.index if means[m] >= means.max() - SE]
best_name = min(within, key=lambda m: summary.loc[m, ("sec/epoch", "mean")])
best_cell, best_H = configs[best_name]
best_epochs = int(round(summary.loc[best_name, ("best_epoch", "mean")]))
print(f"一個標準誤（{SE:.4f}）以內：{within} → 選 {best_name}，refit {best_epochs} epochs")

Xi_trva = np.vstack([Xi_tr, Xi_va])
len_trva = np.concatenate([len_tr, len_va])
y_trva = np.concatenate([y_tr, y_va])
set_seed(0)
net_final = TextRNNTorch(V, cell=best_cell, H=best_H)
train_seq(net_final, Xi_trva, len_trva, y_trva, n_epochs=best_epochs, verbose=False)
tfidf_full = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, token_pattern=r"\S+"), LogisticRegression(C=4, max_iter=3000)).fit(docs_tr + docs_va, y_trva)
Xs_te, ls_te = encode(test_tok, max_len=SUB_LEN)

eval_models = {
    "TF-IDF 1-2-gram + LogReg (train+val)": (tfidf_full.predict_proba([" ".join(t) for t in test_tok]), {"ngram": [1, 2], "C": 4}),
    "GRU (scratch NumPy, 8k subset, last 100 tokens)": (gru_np.predict_proba(Xs_te, ls_te), {"E": 32, "H": 64, "train": 8000, "max_len": 100}),
    "LSTM (PyTorch, last 200 tokens, seed 0)": (predict_proba_seq(torch_models["lstm"], Xi_te, len_te), {"E": 64, "H": 64, "max_len": MAX_LEN}),
    "GRU (PyTorch, last 200 tokens, seed 0)": (predict_proba_seq(torch_models["gru"], Xi_te, len_te), {"E": 64, "H": 64, "max_len": MAX_LEN}),
    f"{best_name} tuned (PyTorch, train+val)": (predict_proba_seq(net_final, Xi_te, len_te), {"cell": best_cell, "E": 64, "H": best_H, "max_len": MAX_LEN, "epochs": best_epochs}),
}
test_scores = {}
for name, (prob, params) in eval_models.items():
    m = classification_metrics(y_test, prob.argmax(axis=1), prob[:, 1])
    m["log_loss"] = round(log_loss(y_test, np.clip(prob, 1e-12, 1), labels=[0, 1]), 4)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "accuracy")

#### 配對 bootstrap：GRU − LSTM 的 test accuracy 差距

1. 每一篇算 $d_j = \mathbb{1}[\text{GRU 對}] - \mathbb{1}[\text{LSTM 對}]$（$+1$、$0$、$-1$）。差距的估計值 $= \bar d$。
2. 有放回地抽 25,000 篇，重算 $\bar d$；重複 2,000 次。
3. 2,000 個 $\bar d$ 的 2.5% 與 97.5% 分位數 = 95% 信賴區間。區間**包含 0** → 在這個測試集上分不出誰比較好。

In [ ]:
correct_gru = eval_models["GRU (PyTorch, last 200 tokens, seed 0)"][0].argmax(axis=1) == y_test
correct_lstm = eval_models["LSTM (PyTorch, last 200 tokens, seed 0)"][0].argmax(axis=1) == y_test
d = correct_gru.astype(float) - correct_lstm.astype(float)
boot_idx = np.random.default_rng(0).integers(0, len(d), size=(2000, len(d)))
boot = d[boot_idx].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"GRU − LSTM 的 test accuracy 差距：{d.mean():+.4f}；95% bootstrap 信賴區間 [{lo:+.4f}, {hi:+.4f}]")
print(f"意見不同的篇數：{int((d != 0).sum())}（GRU 對 LSTM 錯 {int((d > 0).sum())}、LSTM 對 GRU 錯 {int((d < 0).sum())}）；"
      f"「各自算 accuracy」的標準誤約 {np.sqrt(0.87 * 0.13 / len(d)):.4f}")
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(boot, bins=50)
ax.axvline(0, color="k", lw=1)
ax.axvline(lo, color="gray", ls=":")
ax.axvline(hi, color="gray", ls=":")
ax.set_xlabel("bootstrap mean of (GRU correct − LSTM correct)")
plt.show()

---
## 7. 診斷

GRU 和 LSTM 在什麼樣的影評上意見不同？依長度分組比較兩者的準確率；看幾篇意見不同的例子。

In [ ]:
prob_g = eval_models["GRU (PyTorch, last 200 tokens, seed 0)"][0]
prob_l = eval_models["LSTM (PyTorch, last 200 tokens, seed 0)"][0]
test_lens = np.array([len(t) for t in test_tok])
by_len = pd.DataFrame({"GRU": correct_gru, "LSTM": correct_lstm, "bin": pd.cut(test_lens, [0, 100, 200, 300, 500, 800, 5000])}).groupby("bin", observed=True).agg(["mean", "size"])
display(by_len.round(3))
print("兩者機率的相關係數：", round(float(np.corrcoef(prob_g[:, 1], prob_l[:, 1])[0, 1]), 3),
      "；平均兩者（ensemble）的 test accuracy：", round(accuracy_score(y_test, ((prob_g + prob_l) / 2).argmax(axis=1)), 4))


def show(idx: np.ndarray, title: str, k: int = 3) -> None:
    print(f"\n==== {title} ====")
    for i in idx[:k]:
        print(f"[label {y_test[i]}；P(pos) GRU {prob_g[i, 1]:.2f}、LSTM {prob_l[i, 1]:.2f}；{test_lens[i]} 字]")
        print("   ...", " ".join(test_tok[i][-60:]))


rng_show = np.random.default_rng(0)
show(rng_show.permutation(np.flatnonzero(d > 0)), "GRU 對、LSTM 錯")
show(rng_show.permutation(np.flatnonzero(d < 0)), "LSTM 對、GRU 錯")

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：
#     (a) 依 accuracy 排序 TF-IDF、手寫 GRU、LSTM、GRU、tuned。跟 TODO 2 的預測一樣嗎？
#     (b) 配對 bootstrap：GRU − LSTM 的 95% 信賴區間包含 0 嗎？為什麼配對的信賴區間比「各自算 accuracy 再相減」窄？（提示：兩個模型大部分的對錯是一樣的）
# 答：
#
# (2) 意見不同的影評：有多少篇？依長度看，有沒有哪一段長度 GRU 或 LSTM 明顯比較好？ensemble 有幫助嗎？為什麼兩個很像的模型 ensemble 的效果比 LSTM + TF-IDF（第 3 課）小？
# 答：
#
# (3) 第三部分 RNN 三課總結（填表）：
#     |                 | RNN | LSTM | GRU |
#     | 狀態            |     |      |     |
#     | 閘門            |     |      |     |
#     | 梯度的捷徑      |     |      |     |
#     | 循環層參數（E=H=64）|  |      |     |
#     | IMDB val acc    |     |      |     |
# 答：
#
# (4) 三課都沒有贏過詞袋 + LogReg。第四部分的 Seq2Seq＋Attention、第五部分的 Transformer 改變了什麼，讓神經網路在文字上大幅超越詞袋？（先猜：預訓練、注意力、資料量）
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

IMDB（train 20,000／validation 5,000／test 25,000；最後 200 個字）

| 模型 | 做法 | 循環層參數 | val acc（3 seeds 平均 ± std） | test acc |
| --- | --- | --- | --- | --- |
| TF-IDF 1～2-gram + LogReg | | — | | |
| GRU（手寫 NumPy，子集） | | | — | |
| LSTM H=64 | | | | |
| GRU H=64 | | | | |
| GRU（參數量同 LSTM） | | | | — |
| tuned | | | — | |

GRU − LSTM 的 test accuracy 差距（配對 bootstrap 95% CI）：

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ GRU 原版（先重置再乘矩陣）與 PyTorch 版的差異對表達能力的影響
- ☐ Jozefowicz, Zaremba & Sutskever (2015)：上萬種循環單元的架構搜尋
- ☐ 統計檢定：McNemar 檢定、配對 bootstrap 與多重比較

**延伸**
- ☐ Minimal Gated Unit（MGU）、只有一個閘門的變體
- ☐ 用 GRU 做 Seq2Seq 的編碼器／解碼器（第四部分）
- ☐ 用 GRU 做時間序列預測（第六部分）